# Misinformation Models—
Trains all models to date and saves them.

Claude helped on implementation details. Logic and concepts are human generated.

### Import configuration and packages ###

In [ ]:
##TO RUN ON COLAB ONLY

#installs
!pip install tweet-preprocessor==0.5.0 feedparser whoosh iterative-stratification fastapi uvicorn
!pip install nltk spacy

#imports
import sys
from pathlib import Path
from google.colab import drive
import pandas as pd
import numpy as np
import torch
import joblib, pickle, torch.nn as nn

#mount google drive and set path-related variables.
drive.mount('/content/drive')
BASE_DIR=Path("/content/drive/MyDrive/linguistic_markers")
SPRINT_DIR = BASE_DIR / "581_Sprint_4"
SRC_DIR = SPRINT_DIR / "src"
DATA_DIR = BASE_DIR / "data" / "final_splits"
SAVE_DIR = SPRINT_DIR / "saved_models"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Add the SPRINT_DIR to the system path so Python can find modules like 'cnn_baseline'
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(1, str(SPRINT_DIR))
sys.path.insert(2, str(SRC_DIR))
sys.path.insert(3, str(DATA_DIR))
sys.path.insert(4, str(BASE_DIR))

# Define the device for training
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")


import cnn_baseline as cnn
#Running this will import FastText vector file, stored on HuggingFace and is >4gb.
from config import FASTTEXT_PATH, TARGETS, SEED
from preprocess import preprocess
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis, print_report
import random
import builtins
builtins.Path = Path
import cnn_mtl_no_ling as cnn_mtl_pos

### Define model name constants and results lists


In [ ]:
# Model name constants — define once, use everywhere
M_TEXT_CNN                 = "TextCNN"
M_TEXT_CNN_TRANSFER        = "TextCNN Transfer"
M_TEXT_CNN_TRANSFER_SHARED = "TextCNN_Transfer_shared"
M_CNN_MTL_LING             = "TextCNN MTL (POS + linguistic)"
M_CNN_MTL_POS              = "TextCNN MTL (POS)"
M_TEXT_CNN_BOOTSTRAP       = "TextCNN Bootstrap"
M_LOGREG                   = "LogReg"
M_LOGREG_EMBED             = "LogReg (+ embeddings)"
M_LOGREG_MTL_POS           = "LogReg MTL (cascaded POS)"
M_LOGREG_EMBED_BOOTSTRAP   = "LogReg (+ embeddings) Bootstrap"

M_ENSEMBLE_SOFT_VOTE       = "Soft Vote Ensemble"
M_ENSEMBLE_MOTIVATED       = "Motivated Ensemble"


AL_PCTS        = [0.05, 0.10, 0.15, 0.20]
CNN_AL_MODELS  = [f"TextCNN AL {int(p*100)}%" for p in AL_PCTS]
LR_AL_MODELS   = [f"LogReg (+emb) AL {int(p*100)}%" for p in AL_PCTS]

CNN_MODELS  = [M_TEXT_CNN, M_TEXT_CNN_TRANSFER, M_CNN_MTL_LING, M_CNN_MTL_POS,
               M_TEXT_CNN_BOOTSTRAP] + CNN_AL_MODELS
LR_MODELS   = [M_LOGREG, M_LOGREG_EMBED, M_LOGREG_MTL_POS,
               M_LOGREG_EMBED_BOOTSTRAP] + LR_AL_MODELS
BASE_MODELS = LR_MODELS + CNN_MODELS
ENSEMBLE_MODELS = [M_ENSEMBLE_SOFT_VOTE, M_ENSEMBLE_MOTIVATED]
                  
OPINION_LABEL = "opinion_label"
MISINFORM_LABEL = "misinformation_label"

# Each model should produce (preds, labels, probs) and store them in `results`.
# Add new models below following the same pattern.
results = {}

#metrics_cache for later use in ordering all models by Macro F1 and making easy displays
metrics_cache = {}

### Load data

In [ ]:

# Load data (shared across all models)
train_rows = cnn.load_csv(DATA_DIR / "mis_df_train.csv")
dev_rows   = cnn.load_csv(DATA_DIR / "mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

### Utility functions for saving check points, reproducibiity, displaying metrics, ranking metrics

In [ ]:

def rank_key(metrics_key):
    """Sort key for model selection: primary = Macro F1, tiebreaker = AUC-ROC.
    Returns (-inf, -inf) for models not trained on this target, excluding them from selection."""
    m = metrics_cache.get(metrics_key)
    if m is None:
        return (float("-inf"), float("-inf"))
    return (m["macro_f1"], m.get("auc_roc", 0.0))


def display_metrics(key_prefix):
    """Display a metrics table for one ensemble/model prefix across all TARGETS."""
    pd.set_option("display.float_format", "{:.4f}".format)
    rows = []
    for t in TARGETS:
        key = f"{key_prefix} — {t}"
        m = metrics_cache[key]
        rows.append({
            "Model": key,
            "Macro F1": m["macro_f1"],
            "F1 (not-op)": m["f1_class0"],
            "F1 (opinion)": m["f1_class1"],
            "F1.5 (recall-weighted)": m["fbeta_class1"],
            "AUC-ROC": m.get("auc_roc", float("nan")),
        })
    return pd.DataFrame(rows).set_index("Model")

def make_reproducible():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.use_deterministic_algorithms(True, warn_only=True)

def save_checkpoint(key, model=None, result=None):
    """Save model weights and/or (preds, labels, probs) result tuple."""
    safe_key = key.replace(" ", "_").replace("/", "-").replace("(", "").replace(")", "").replace("+", "plus")
    saved = []

    if model is not None:
        if isinstance(model, nn.Module):
            torch.save(model.state_dict(), SAVE_DIR / f"{safe_key}.pt")
            saved.append("state_dict")
        else:
            joblib.dump(model, SAVE_DIR / f"{safe_key}.joblib")
            saved.append("joblib")

    if result is not None:
        with open(SAVE_DIR / f"{safe_key}_result.pkl", "wb") as f:
            pickle.dump(result, f)
        saved.append("result")

    if saved:
        print(f"  [saved] {safe_key} ({', '.join(saved)})")


### Create and train models


#### CNN Baseline

In [ ]:
#CNN Baseline

make_reproducible()

vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)


cnn_model_opinion = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_model_opinion = cnn.train_model(cnn_model_opinion, train_loader, dev_loader, train_rows, DEVICE,
                                    train_targets=[OPINION_LABEL])
key = f"{M_TEXT_CNN} — {OPINION_LABEL}"
results[key] = cnn.predict(cnn_model_opinion, dev_loader, DEVICE, target=OPINION_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_model_opinion, result=results[key])

cnn_model_misinform = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_model_misinform = cnn.train_model(cnn_model_misinform, train_loader, dev_loader, train_rows, DEVICE,
                                      train_targets=[MISINFORM_LABEL])
key = f"{M_TEXT_CNN} — {MISINFORM_LABEL}"
results[key] = cnn.predict(cnn_model_misinform, dev_loader, DEVICE, target=MISINFORM_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_model_misinform, result=results[key])


#### CNN Transfer Learning

In [ ]:
# CNN Transfer Learning
cnn_tf_model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_tf_model = cnn.train_model(cnn_tf_model, train_loader, dev_loader, train_rows, DEVICE)
save_checkpoint(M_TEXT_CNN_TRANSFER_SHARED, model=cnn_tf_model)

for target in TARGETS:
    key = f"{M_TEXT_CNN_TRANSFER} — {target}"
    results[key] = cnn.predict(cnn_tf_model, dev_loader, DEVICE, target=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])

#### CNN MTL — POS & Linguistic Features


In [ ]:
import cnn_mtl_ling as cnn_mtl_ling

train_loader_ling = cnn_mtl_ling.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader_ling   = cnn_mtl_ling.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

cnn_mtl_l_model_opinion = cnn_mtl_ling.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_mtl_l_model_opinion = cnn_mtl_ling.train_model(cnn_mtl_l_model_opinion, train_loader_ling,
                                                    dev_loader_ling, train_rows, DEVICE,
                                                    train_targets=[OPINION_LABEL])
key = f"{M_CNN_MTL_LING} — {OPINION_LABEL}"
results[key] = cnn_mtl_ling.predict(cnn_mtl_l_model_opinion, dev_loader_ling, DEVICE, target=OPINION_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_mtl_l_model_opinion, result=results[key])

cnn_mtl_l_model_misinform = cnn_mtl_ling.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_mtl_l_model_misinform = cnn_mtl_ling.train_model(cnn_mtl_l_model_misinform, train_loader_ling,
                                                      dev_loader_ling, train_rows, DEVICE,
                                                      train_targets=[MISINFORM_LABEL])
key = f"{M_CNN_MTL_LING} — {MISINFORM_LABEL}"
results[key] = cnn_mtl_ling.predict(cnn_mtl_l_model_misinform, dev_loader_ling, DEVICE, target=MISINFORM_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_mtl_l_model_misinform, result=results[key])


#### CNN MTL - POS Features only


In [ ]:
import cnn_mtl_no_ling as cnn_mtl_pos

train_loader_pos = cnn_mtl_pos.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader_pos   = cnn_mtl_pos.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

cnn_mtl_p_model_opinion = cnn_mtl_pos.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_mtl_p_model_opinion = cnn_mtl_pos.train_model(cnn_mtl_p_model_opinion, train_loader_pos,
                                                   dev_loader_pos, train_rows, DEVICE,
                                                   train_targets=[OPINION_LABEL])
key = f"{M_CNN_MTL_POS} — {OPINION_LABEL}"
results[key] = cnn_mtl_pos.predict(cnn_mtl_p_model_opinion, dev_loader_pos, DEVICE, target=OPINION_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_mtl_p_model_opinion, result=results[key])

cnn_mtl_p_model_misinform = cnn_mtl_pos.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_mtl_p_model_misinform = cnn_mtl_pos.train_model(cnn_mtl_p_model_misinform, train_loader_pos,
                                                     dev_loader_pos, train_rows, DEVICE,
                                                     train_targets=[MISINFORM_LABEL])
key = f"{M_CNN_MTL_POS} — {MISINFORM_LABEL}"
results[key] = cnn_mtl_pos.predict(cnn_mtl_p_model_misinform, dev_loader_pos, DEVICE, target=MISINFORM_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_mtl_p_model_misinform, result=results[key])


#### Logistic Regression Baseline

In [ ]:
import logreg_baseline as lr

for target in TARGETS:
    key = f"{M_LOGREG} — {target}"
    results[key] = lr.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])

#### Logistic Regression Transfer Learning

In [ ]:
import logreg_transfer as lr_transfer

for target in TARGETS:
    key = f"{M_LOGREG_EMBED} — {target}"
    results[key] = lr_transfer.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])

#### LogReg MTL — Cascaded POS Prediction 


In [ ]:
import logreg_mtl as lr_mtl

for target in TARGETS:
    key = f"{M_LOGREG_MTL_POS} — {target}"
    results[key] = lr_mtl.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])

## Bootstrapping


*   Use TextCNN for Opinion
*   Use LogReg(+embeddings) for Misinformation

In [ ]:
# Reproducibility — reset RNG state
make_reproducible()


# import 250 bootstrap rows with just id, text, and labels so predict function can be used
bootstrap_rows = pd.read_csv(DATA_DIR / "additional_250_annotated.csv", usecols=range(4))
bootstrap_records = bootstrap_rows.to_dict('records')

bootstrap_loader = cnn.make_loader(bootstrap_records, vocab, shuffle=False, tokenize_fn=preprocess)

# bootstrap opinion — reuse already-trained cnn_model_opinion
bootstrap_opinion_preds, _, _ = cnn.predict(cnn_model_opinion, bootstrap_loader, DEVICE, target=OPINION_LABEL)

# bootstrap misinfo
bootstrap_misinfo_preds, _, bootstrap_misinfo_probs = lr_transfer.run(train_rows, bootstrap_records, task=MISINFORM_LABEL)

# bootstrap combined preds
bootstrap_combined_preds = pd.DataFrame({
    "id": bootstrap_rows["id"],
    "text": bootstrap_rows["text"],
    MISINFORM_LABEL: bootstrap_misinfo_preds,
    OPINION_LABEL: bootstrap_opinion_preds
})
bootstrap_combined_preds.to_csv(DATA_DIR / "bootstrap_combined_preds.csv", index=False)

# retrain and re-evaluate opinion cnn on bootstrap-augmented data
more_train_rows = train_rows + bootstrap_combined_preds.to_dict('records')
bootstrap_train_loader = cnn.make_loader(more_train_rows, vocab, shuffle=True, tokenize_fn=preprocess)

cnn_bs_model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_bs_model = cnn.train_model(cnn_bs_model, bootstrap_train_loader, dev_loader, dev_rows, DEVICE,
                               train_targets=[OPINION_LABEL])
key = f"{M_TEXT_CNN_BOOTSTRAP} — {OPINION_LABEL}"
results[key] = cnn.predict(cnn_bs_model, dev_loader, DEVICE, target=OPINION_LABEL)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, model=cnn_bs_model, result=results[key])

# retrain and re-evaluate misinformation logreg on bootstrap-augmented data
target = MISINFORM_LABEL
key = f"{M_LOGREG_EMBED_BOOTSTRAP} — {target}"
results[key] = lr_transfer.run(more_train_rows, dev_rows, task=target)
metrics_cache[key] = compute_metrics(*results[key])
save_checkpoint(key, result=results[key])


## Active Learning

We used the 250 bootstrap examples tagged by the model and ranked them by an active learning uncertainty score.

### Use bootstrap to collect probabilities

In [ ]:

# Reuse already-trained cnn_model_opinion — no retraining needed
bs_op_preds, _, bs_op_probs = cnn.predict(cnn_model_opinion, bootstrap_loader, DEVICE, target=OPINION_LABEL)

# Reuse misinformation LogReg predictions from bootstrap cell — no retraining needed
bs_mis_preds, bs_mis_probs = bootstrap_misinfo_preds, bootstrap_misinfo_probs

# LogReg opinion predictions for committee disagreement scoring (COLX_581 Lecture 7)
bs_op_preds_lr, _, bs_op_probs_lr = lr_transfer.run(train_rows, bootstrap_records,
                                                     task=OPINION_LABEL)
bs_mis_preds_lr = bs_mis_preds

print(f"Opinion probs shape:  {np.array(bs_op_probs).shape}")
print(f"Misinfo probs shape:  {np.array(bs_mis_probs).shape}")

### Uncertainty and committee disagreement scores

Combined score averages both signals

In [ ]:
op_probs  = np.array(bs_op_probs,  dtype=float)
mis_probs = np.array(bs_mis_probs, dtype=float)

# Uncertainty scores
op_uncertainty  = 1.0 - np.maximum(op_probs,  1.0 - op_probs)
mis_uncertainty = 1.0 - np.maximum(mis_probs, 1.0 - mis_probs)
avg_uncertainty = (op_uncertainty + mis_uncertainty) / 2.0

# Committee disagreement: (1) CNN and LogReg prediction disagreement; (0) agreement
op_disagreement  = (np.array(bs_op_preds) != np.array(bs_op_preds_lr)).astype(float)
mis_disagreement = (np.array(bs_mis_preds) != np.array(bs_mis_preds_lr)).astype(float)
avg_disagreement = (op_disagreement + mis_disagreement) / 2.0

# Combined score (average of uncertainty and committee disagreement)
# Questioned Claude on best approach for scoring our specific dataset
combined_score = (avg_uncertainty + avg_disagreement) / 2.0

# Claude assisted with code and approach
al_df = bootstrap_rows[["id", "text"]].copy()
al_df["bs_opinion_pred"]  = bs_op_preds
al_df["bs_misinfo_pred"]  = bs_mis_preds
al_df["op_prob"]          = op_probs
al_df["mis_prob"]         = mis_probs
al_df["op_uncertainty"]   = op_uncertainty
al_df["mis_uncertainty"]  = mis_uncertainty
al_df["avg_uncertainty"]  = avg_uncertainty
al_df["op_disagreement"]  = op_disagreement
al_df["mis_disagreement"] = mis_disagreement
al_df["avg_disagreement"] = avg_disagreement
al_df["combined_score"]   = combined_score

al_df_ranked = al_df.sort_values("combined_score", ascending=False).reset_index(drop=True)
al_df_ranked.to_csv(DATA_DIR / "active_learning_ranked.csv", index=False)

print(f"Saved active_learning_ranked.csv ({len(al_df_ranked)} rows)")
print(f"\nTop 10 most uncertain/disagreed examples:")
al_df_ranked[["id", "combined_score", "avg_uncertainty", "avg_disagreement"]].head(10)

### Manual Re-annotation

The file `active_learning_ranked.csv` lists all 250 bootstrap examples.

**Manual annotation task:** Re-label the top 20% by hand using the same guidelines as the original dataset. Save the result as `active_learning_annotated.csv` with columns `id`, `text`, `misinformation_label`, `opinion_label`. (Completed by Shiao-li last week).

The accumulation test below uses human labels for the top-k rows and model labels for the rest.

In [ ]:
cnn_al_models = {}

#Claude assisted with fixing code errors
def run_accumulation_test(pct, al_annotated_df, bootstrap_model_df):
    """
    pct: fraction of 250 to use with human labels
    al_annotated_df: active_learning_annotated.csv (human labels in top rows)
    bootstrap_model_df: bootstrap_combined_preds.csv (model labels for all 250)
    """
    al_idx = AL_PCTS.index(pct)
    n_top = int(round(len(al_annotated_df) * pct))
    print(f"AL {pct*100:.0f}%: {n_top} human-annotated + {len(al_annotated_df)-n_top} model-labelled")

    top_k = al_annotated_df.iloc[:n_top][["id", "text",
                                          MISINFORM_LABEL, OPINION_LABEL]].copy()
    remaining_ids = set(al_annotated_df.iloc[n_top:]["id"])
    rest = bootstrap_model_df[bootstrap_model_df["id"].isin(remaining_ids)].copy()

    extra_records  = pd.concat([top_k, rest], ignore_index=True).to_dict('records')
    augmented_rows = train_rows + extra_records
    aug_loader     = cnn.make_loader(augmented_rows, vocab, shuffle=True, tokenize_fn=preprocess)

    # Retrain Opinion CNN
    model_op = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
    model_op = cnn.train_model(model_op, aug_loader, dev_loader,
                               dev_rows, DEVICE, train_targets=[OPINION_LABEL])
    res_op  = cnn.predict(model_op, dev_loader, DEVICE, target=OPINION_LABEL)
    m_op    = compute_metrics(*res_op)
    key_op  = f"{CNN_AL_MODELS[al_idx]} — {OPINION_LABEL}"
    results[key_op] = res_op
    metrics_cache[key_op] = m_op
    save_checkpoint(key_op, model=model_op, result=res_op)
    cnn_al_models[pct] = model_op

    # Retrain Misinformation LogReg
    res_mis = lr_transfer.run(augmented_rows, dev_rows, task=MISINFORM_LABEL)
    m_mis   = compute_metrics(*res_mis)
    key_mis = f"{LR_AL_MODELS[al_idx]} — {MISINFORM_LABEL}"
    results[key_mis] = res_mis
    metrics_cache[key_mis] = m_mis
    save_checkpoint(key_mis, result=res_mis)

    print(f"Opinion  Macro F1: {m_op['macro_f1']:.4f}  F1-op: {m_op['f1_class1']:.4f}  "
          f"AUC: {m_op.get('auc_roc', float('nan')):.4f}")
    print(f"Misinfo  Macro F1: {m_mis['macro_f1']:.4f}  F1-mis: {m_mis['f1_class1']:.4f}  "
          f"AUC: {m_mis.get('auc_roc', float('nan')):.4f}")
    return {"opinion": m_op, "misinfo": m_mis, "n_human": n_top}


### Create annotated active learning dataset

In [ ]:
ranked = pd.read_csv(DATA_DIR / "active_learning_ranked.csv")
annotated = pd.read_csv(DATA_DIR / "additional_250_annotated.csv")

al_annotated = ranked.merge(
    annotated[["id", MISINFORM_LABEL, OPINION_LABEL]],
    on="id",
    how="left"
)
al_annotated.to_csv(DATA_DIR / "active_learning_annotated.csv", index=False)

### Load annotated file and run accumulation test

In [ ]:
make_reproducible()

al_annotated_df    = pd.read_csv(DATA_DIR / "active_learning_annotated.csv")
bootstrap_model_df = pd.read_csv(DATA_DIR / "bootstrap_combined_preds.csv")

accum_results = {}
for pct in AL_PCTS:
    accum_results[pct] = run_accumulation_test(pct, al_annotated_df, bootstrap_model_df)


### Active learning summary table

In [ ]:
summary_rows = []
for pct, res in accum_results.items():
    summary_rows.append({
        "AL threshold":      f"{pct*100:.0f}% ({res['n_human']} examples)",
        "Opinion Macro F1": res["opinion"]["macro_f1"],
        "Opinion F1 (op)":  res["opinion"]["f1_class1"],
        "Opinion AUC":      res["opinion"].get("auc_roc", float("nan")),
        "Misinfo Macro F1": res["misinfo"]["macro_f1"],
        "Misinfo F1 (mis)": res["misinfo"]["f1_class1"],
        "Misinfo AUC":      res["misinfo"].get("auc_roc", float("nan")),
    })

pd.set_option("display.float_format", "{:.4f}".format)
summary_df = pd.DataFrame(summary_rows).set_index("AL threshold")
print("Active Learning Accumulation Test")
print("─" * 70)
display(summary_df)

### Ensembles


In [ ]:
import importlib
import simple_ensemble, motivated_ensemble
importlib.reload(simple_ensemble)
importlib.reload(motivated_ensemble)
from simple_ensemble import soft_vote
from motivated_ensemble import motivated_soft_vote

# Ensemble soft vote: best LogReg + best CNN per task (selected dynamically from metrics_cache)
best_cnn = {
    task: max(CNN_MODELS, key=lambda m: rank_key(f"{m} — {task}"))
    for task in TARGETS
}
best_logreg = {
    task: max(LR_MODELS, key=lambda m: rank_key(f"{m} — {task}"))
    for task in TARGETS
}

for task in TARGETS:
    print(f"{task}: best LogReg = {best_logreg[task]} | best CNN = {best_cnn[task]}")

for task in TARGETS:
    preds, labels, probs = soft_vote([
        results[f"{best_logreg[task]} — {task}"],
        results[f"{best_cnn[task]} — {task}"],
    ])
    key = f"{M_ENSEMBLE_SOFT_VOTE} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_SOFT_VOTE)

In [ ]:
# Motivated ensemble: best LogReg + best CNN, F1-weighted with threshold sweep
for task in TARGETS:
    ensemble_models = [best_logreg[task], best_cnn[task]]
    f1_weights = [metrics_cache[f"{m} — {task}"]["macro_f1"] for m in ensemble_models]
    preds, labels, probs = motivated_soft_vote(
        model_outputs=[results[f"{m} — {task}"] for m in ensemble_models],
        f1_weights=f1_weights,
    )
    key = f"{M_ENSEMBLE_MOTIVATED} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_MOTIVATED)


### All Results by Target

In [ ]:
# All results by target — everything is now in metrics_cache
all_rows = [{"Model": name, "Macro F1": m["macro_f1"],
             "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
             "F1.5 (recall-weighted)": m["fbeta_class1"],
             "AUC-ROC": m.get("auc_roc", float("nan"))}
            for name, m in metrics_cache.items()]

all_df = pd.DataFrame(all_rows)

for task, label in [("Opinion", OPINION_LABEL), ("Misinformation", MISINFORM_LABEL)]:
    mask = all_df["Model"].str.endswith(f"— {label}")
    df = (all_df[mask]
          .copy()
          .assign(Model=lambda d: d["Model"].str.replace(f" — {label}", "", regex=False))
          .set_index("Model")
          .sort_values("Macro F1", ascending=False))
    print(task, "Ordered by Macro F1 (Desc)")
    print("─" * 60)
    display(df)
    print()

### Best Model Analysis: Confusion Matrix & Examples

In [ ]:
def print_quadrant_examples(dev_rows, preds, labels, n=3):
    """Print up to n examples from each confusion matrix quadrant."""
    quadrants = {
        "True Positives  (predicted=1, actual=1)": [],
        "True Negatives  (predicted=0, actual=0)": [],
        "False Positives (predicted=1, actual=0)": [],
        "False Negatives (predicted=0, actual=1)": [],
    }
    for row, p, l in zip(dev_rows, preds, labels):
        if   p == 1 and l == 1: quadrants["True Positives  (predicted=1, actual=1)"].append(row)
        elif p == 0 and l == 0: quadrants["True Negatives  (predicted=0, actual=0)"].append(row)
        elif p == 1 and l == 0: quadrants["False Positives (predicted=1, actual=0)"].append(row)
        elif p == 0 and l == 1: quadrants["False Negatives (predicted=0, actual=1)"].append(row)

    for label, rows in quadrants.items():
        print(f"\n── {label} ({len(rows)} total, showing {min(n, len(rows))}) ──")
        for r in rows[:n]:
            print(f"  [{r['id']}] {r['text'][:140]!r}")


for task, label in [("Opinion", OPINION_LABEL), ("Misinformation", MISINFORM_LABEL)]:
    task_keys = [k for k in metrics_cache if k.endswith(f"— {label}")]
    best_key  = max(task_keys, key=rank_key)
    preds, labels_list, probs = results[best_key]
    m = metrics_cache[best_key]

    print(f"\n{'='*60}")
    print(f"{task} — best model: {best_key.replace(f' — {label}', '')}")
    print(f"Macro F1: {m['macro_f1']:.4f}  |  AUC-ROC: {m.get('auc_roc', float('nan')):.4f}")
    print(f"{'='*60}")
    print_confusion_matrix(preds, labels_list)
    print_quadrant_examples(dev_rows, preds, labels_list, n=8)